# Fashion Retrieval — Evaluation (Config A and B)

Evaluates retrieval performance for Config A and Config B using frozen CLIP.

---

### Metrics
| Metric | Description |
|--------|-------------|
| Recall@K | 1 if any correct item in top-K results, else 0 |
| NDCG@K | Position-aware gain — correct items ranked earlier = higher score |
| mAP@K | Mean average precision — rewards finding all correct items early |

K ∈ {5, 10, 15}

---

### How it works
1. Encode query images with CLIP (image-only — no caption for query)
2. Search gallery FAISS index → top-K hits
3. Check hits against ground truth (same item_id = correct)
4. Compute metrics per query, average over seeds

---

### Inputs
- `vr-yolo-bbox-cropped-images` — query crops + master_crops.csv
- clip indexes dataset — FAISS indexes + item_index_map.csv

## 1. Install Packages

In [1]:
!pip uninstall -y faiss faiss-gpu
!pip install ftfy regex transformers faiss-cpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.1 MB/s eta 0:00:00


## 2. Imports

In [2]:
import os
import json
import numpy as np
import pandas as pd
import torch
import faiss
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
import warnings
warnings.filterwarnings('ignore')

GPU = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Runtime device : {GPU}')
print('Imports complete!')

Runtime device : cuda
Imports complete!


## 3. Configuration

**Change `ROLL_SEEDS` to your team's actual roll numbers.**

In [3]:
# ============================================================
#  SET YOUR TEAM ROLL NUMBERS AS SEEDS
# ============================================================
ROLL_SEEDS = [42, 55, 63, 71]   # replace with actual roll numbers
# ============================================================

BBOX_CROPS_DIR  = '/kaggle/input/datasets/akibatra25/vr-yolo-bbox-cropped-images'
INDEXES_DIR     = '/kaggle/input/datasets/akibatra25/clip-ab-output'  # update after saving

K_LIST      = [5, 10, 15]
ENC_BATCH   = 64
CLIP_CKPT   = 'openai/clip-vit-base-patch32'

# Configs: (name, index_filename, beta)
EVAL_CONFIGS = [
    ('Config_A_beta1.0', 'idx_A_b10.bin', 1.0),
    ('Config_B_beta0.7', 'idx_B_b07.bin', 0.7),
    ('Config_B_beta0.5', 'idx_B_b05.bin', 0.5),
]

for tag, p in [('BBOX_CROPS_DIR', BBOX_CROPS_DIR), ('INDEXES_DIR', INDEXES_DIR)]:
    ok = 'Found ✓' if os.path.exists(p) else 'NOT FOUND ✗'
    print(f'[{ok}] {tag}')

print(f'\nRoll seeds : {ROLL_SEEDS}')
print(f'K values   : {K_LIST}')

[Found ✓] BBOX_CROPS_DIR
[Found ✓] INDEXES_DIR

Roll seeds : [42, 55, 63, 71]
K values   : [5, 10, 15]


## 4. Load Query Data and Gallery Metadata

In [4]:
full_table = pd.read_csv(os.path.join(BBOX_CROPS_DIR, 'master_crops.csv'))
qry_table  = full_table[full_table['split'] == 'query'].reset_index(drop=True)

def translate_path(saved_path):
    if pd.isna(saved_path): return saved_path
    for pfx in ['/kaggle/working/', '/kaggle/input/']:
        if saved_path.startswith(pfx):
            tail = saved_path.replace(pfx, '')
            for ds in ['vr-yolo-bbox-cropped-images/', 'datasets/akibatra25/vr-yolo-bbox-cropped-images/']:
                tail = tail.replace(ds, '')
            return os.path.join(BBOX_CROPS_DIR, tail)
    return saved_path

qry_table['img_path'] = qry_table['crop_path'].apply(translate_path)
qry_table['on_disk']  = qry_table['img_path'].apply(
    lambda p: os.path.exists(p) if isinstance(p, str) else False
)

if qry_table['on_disk'].sum() < len(qry_table) * 0.9:
    def direct_path(img_name):
        rel = img_name[4:] if img_name.startswith('img/') else img_name
        for sub in ['data/bbox_crops', 'data/yolo_crops']:
            p = os.path.join(BBOX_CROPS_DIR, sub, rel)
            if os.path.exists(p): return p
        return os.path.join(BBOX_CROPS_DIR, 'data/bbox_crops', rel)
    qry_table['img_path'] = qry_table['image_name'].apply(direct_path)
    qry_table['on_disk']  = qry_table['img_path'].apply(os.path.exists)

valid_qry = qry_table[qry_table['on_disk']].reset_index(drop=True)
item_map  = pd.read_csv(os.path.join(INDEXES_DIR, 'item_index_map.csv'))

print(f'Valid query images  : {len(valid_qry):,}')
print(f'Gallery index rows  : {len(item_map):,}')
print(f'Unique query items  : {valid_qry["item_id"].nunique():,}')

Valid query images  : 14,218
Gallery index rows  : 12,612
Unique query items  : 3,985


## 5. Load CLIP and Encode All Query Images

In [5]:
print(f'Loading CLIP: {CLIP_CKPT}')
clip_proc = CLIPProcessor.from_pretrained(CLIP_CKPT)
clip_net  = CLIPModel.from_pretrained(CLIP_CKPT).to(GPU)
for p in clip_net.parameters():
    p.requires_grad = False
clip_net.eval()
VEC_DIM = clip_net.config.projection_dim
print(f'CLIP loaded! Vector dim: {VEC_DIM}')

# Encode all query images — same for every config (image-only, no caption)
n_qry     = len(valid_qry)
qry_vecs  = np.zeros((n_qry, VEC_DIM), dtype=np.float32)

print(f'Encoding {n_qry:,} query images...')
for s in tqdm(range(0, n_qry, ENC_BATCH), desc='Query encoding'):
    chunk   = valid_qry.iloc[s : s + ENC_BATCH]
    imgs, ok_idx = [], []
    for i, (_, row) in enumerate(chunk.iterrows()):
        try:
            imgs.append(Image.open(row['img_path']).convert('RGB'))
            ok_idx.append(i)
        except Exception: pass
    if not imgs: continue
    inp = clip_proc(images=imgs, return_tensors='pt', padding=True).to(GPU)
    with torch.no_grad():
        raw  = clip_net.get_image_features(**inp)
        vecs = raw.pooler_output if hasattr(raw, 'pooler_output') and not isinstance(raw, torch.Tensor) else raw
    vecs = vecs / vecs.norm(dim=-1, keepdim=True)
    for li, gi in enumerate(ok_idx):
        qry_vecs[s + gi] = vecs[li].cpu().numpy()

print(f'Query vectors shape: {qry_vecs.shape}')

Loading CLIP: openai/clip-vit-base-patch32


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIP loaded! Vector dim: 512
Encoding 14,218 query images...


Query encoding: 100%|██████████| 223/223 [03:35<00:00,  1.03it/s]

Query vectors shape: (14218, 512)


## 6. Metric Functions

In [6]:
def compute_recall(retrieved, q_id, k):
    return int(any(r == q_id for r in retrieved[:k]))


def compute_ndcg(retrieved, q_id, n_relevant, k):
    dcg = 0.0
    for rank, rid in enumerate(retrieved[:k], start=1):
        if rid == q_id:
            dcg += 1.0 / np.log2(rank + 1)
    ideal_hits = min(n_relevant, k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0


def compute_ap(retrieved, q_id, n_relevant, k):
    hits = 0
    running_prec = 0.0
    for rank, rid in enumerate(retrieved[:k], start=1):
        if rid == q_id:
            hits += 1
            running_prec += hits / rank
    denom = min(n_relevant, k)
    return running_prec / denom if denom > 0 else 0.0


print('Metric functions ready ✓')
_test_ret = ['A', 'B', 'A', 'C', 'A']
assert compute_recall(_test_ret, 'A', 5) == 1
print(f'Test Recall@5 : {compute_recall(_test_ret, "A", 5)}  ✓')

Metric functions ready ✓
Test Recall@5 : 1  ✓


## 7. Run Evaluation

In [7]:
gal_item_ids    = item_map['item_id'].tolist()
gal_item_counts = item_map['item_id'].value_counts().to_dict()
result_rows = []

for cfg_name, idx_file, beta in EVAL_CONFIGS:
    print(f'\n=== {cfg_name} ===')

    idx_path = os.path.join(INDEXES_DIR, idx_file)
    srch_idx = faiss.read_index(idx_path)
    print(f'  Index: {srch_idx.ntotal:,} vectors')

    per_seed = {k: {'recall': [], 'ndcg': [], 'ap': []} for k in K_LIST}

    for seed in ROLL_SEEDS:
        np.random.seed(seed)
        torch.manual_seed(seed)

        n_samp   = min(max(500, int(0.2 * len(valid_qry))), len(valid_qry))
        s_idx    = np.random.choice(len(valid_qry), n_samp, replace=False)
        s_vecs   = qry_vecs[s_idx].astype(np.float32)
        s_ids    = valid_qry.iloc[s_idx]['item_id'].tolist()

        _, hits  = srch_idx.search(s_vecs, 16)

        rec  = {k: [] for k in K_LIST}
        ndcg = {k: [] for k in K_LIST}
        aps  = {k: [] for k in K_LIST}

        for q_id, hit_row in zip(s_ids, hits):
            retrieved = [gal_item_ids[i] for i in hit_row if i < len(gal_item_ids)]
            n_rel     = gal_item_counts.get(q_id, 1)
            for k in K_LIST:
                rec[k].append(compute_recall(retrieved, q_id, k))
                ndcg[k].append(compute_ndcg(retrieved, q_id, n_rel, k))
                aps[k].append(compute_ap(retrieved, q_id, n_rel, k))

        for k in K_LIST:
            per_seed[k]['recall'].append(np.mean(rec[k]))
            per_seed[k]['ndcg'].append(np.mean(ndcg[k]))
            per_seed[k]['ap'].append(np.mean(aps[k]))

        print(f'  Seed {seed}: R@10={np.mean(rec[10]):.4f}  NDCG@10={np.mean(ndcg[10]):.4f}  mAP@10={np.mean(aps[10]):.4f}')

    for k in K_LIST:
        result_rows.append({
            'config': cfg_name, 'K': k,
            'Recall_mean': np.mean(per_seed[k]['recall']),
            'Recall_std' : np.std(per_seed[k]['recall']),
            'NDCG_mean'  : np.mean(per_seed[k]['ndcg']),
            'NDCG_std'   : np.std(per_seed[k]['ndcg']),
            'mAP_mean'   : np.mean(per_seed[k]['ap']),
            'mAP_std'    : np.std(per_seed[k]['ap']),
        })

print('\nEvaluation complete!')


=== Config_A_beta1.0 ===
  Index: 12,612 vectors
  Seed 42: R@10=0.5733  NDCG@10=0.2443  mAP@10=0.1709
  Seed 55: R@10=0.5593  NDCG@10=0.2377  mAP@10=0.1655
  Seed 63: R@10=0.5568  NDCG@10=0.2393  mAP@10=0.1676
  Seed 71: R@10=0.5670  NDCG@10=0.2408  mAP@10=0.1679

=== Config_B_beta0.7 ===
  Index: 12,612 vectors
  Seed 42: R@10=0.5663  NDCG@10=0.2411  mAP@10=0.1697
  Seed 55: R@10=0.5603  NDCG@10=0.2367  mAP@10=0.1650
  Seed 63: R@10=0.5501  NDCG@10=0.2387  mAP@10=0.1683
  Seed 71: R@10=0.5638  NDCG@10=0.2366  mAP@10=0.1649

=== Config_B_beta0.5 ===
  Index: 12,612 vectors
  Seed 42: R@10=0.5237  NDCG@10=0.2101  mAP@10=0.1440
  Seed 55: R@10=0.5188  NDCG@10=0.2063  mAP@10=0.1411
  Seed 63: R@10=0.5139  NDCG@10=0.2081  mAP@10=0.1429
  Seed 71: R@10=0.5146  NDCG@10=0.2063  mAP@10=0.1410

Evaluation complete!


## 8. Results Table

In [8]:
results_df = pd.DataFrame(result_rows)

print('\n=== ABLATION RESULTS (Config A and B) ===')
print(f'Seeds: {ROLL_SEEDS}  |  Format: mean ± std')
print()

for cfg_name, _, _ in EVAL_CONFIGS:
    rows = results_df[results_df['config'] == cfg_name]
    print(f'Config: {cfg_name}')
    print(f'  {"K":>4}  {"Recall@K":>14}  {"NDCG@K":>14}  {"mAP@K":>14}')
    print(f'  {"-"*52}')
    for _, r in rows.iterrows():
        print(f'  K={int(r["K"]):>2}  '
              f'{r["Recall_mean"]:.4f}±{r["Recall_std"]:.4f}  '
              f'{r["NDCG_mean"]:.4f}±{r["NDCG_std"]:.4f}  '
              f'{r["mAP_mean"]:.4f}±{r["mAP_std"]:.4f}')
    print()

results_df.to_csv('/kaggle/working/eval_results_AB.csv', index=False)
print('Results saved to eval_results_AB.csv')


=== ABLATION RESULTS (Config A and B) ===
Seeds: [42, 55, 63, 71]  |  Format: mean ± std

Config: Config_A_beta1.0
     K        Recall@K          NDCG@K           mAP@K
  ----------------------------------------------------
  K= 5  0.4970±0.0089  0.2376±0.0027  0.1763±0.0016
  K=10  0.5641±0.0065  0.2405±0.0024  0.1680±0.0019
  K=15  0.6020±0.0063  0.2469±0.0025  0.1679±0.0020

Config: Config_B_beta0.7
     K        Recall@K          NDCG@K           mAP@K
  ----------------------------------------------------
  K= 5  0.4839±0.0052  0.2321±0.0019  0.1731±0.0016
  K=10  0.5601±0.0062  0.2383±0.0018  0.1670±0.0021
  K=15  0.6024±0.0051  0.2457±0.0016  0.1676±0.0020

Config: Config_B_beta0.5
     K        Recall@K          NDCG@K           mAP@K
  ----------------------------------------------------
  K= 5  0.4317±0.0054  0.1991±0.0016  0.1462±0.0010
  K=10  0.5178±0.0039  0.2077±0.0015  0.1422±0.0013
  K=15  0.5658±0.0063  0.2161±0.0017  0.1434±0.0013

Results saved to eval_results_AB.